In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

In [2]:
data = sns.load_dataset("penguins")
print("Shape:", data.shape)

Shape: (344, 7)


In [5]:
data.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [6]:
print("Duplicated rows:", data.duplicated().sum())

Duplicated rows: 0


In [7]:
rows_before = len(data)
data = data.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", rows_before - len(data))
print("Shape:", data.shape)

Duplicate rows removed: 0
Shape: (344, 7)


In [8]:
missing_percentage = data.isnull().mean() * 100
print(missing_percentage.sort_values(ascending=False).round(2))

sex                  3.20
bill_depth_mm        0.58
bill_length_mm       0.58
flipper_length_mm    0.58
body_mass_g          0.58
island               0.00
species              0.00
dtype: float64


In [9]:
missing_threshold = 70
cols_to_remove = missing_percentage[missing_percentage >= missing_threshold].index.tolist()
data = data.drop(columns=cols_to_remove)
print("Columns removed:", cols_to_remove)
print("Shape:", data.shape)

Columns removed: []
Shape: (344, 7)


In [10]:
complete_data = data.dropna()
measure_data = data.dropna(subset=["bill_length_mm"])

pd.DataFrame({
    "Rows kept": [len(data), len(complete_data), len(measure_data)],
    "Rows removed": [0, len(data) - len(complete_data), len(data) - len(measure_data)],
}, index=[
    "Before row removal",
    "Remove every incomplete row",
    "Remove rows missing bill_length_mm",
])

,Rows kept,Rows removed
Before row removal,344,0
Remove every incomplete row,333,11
Remove rows missing bill_length_mm,342,2


In [11]:
measure_cols = data.select_dtypes(include=np.number).columns.tolist()
n_missing_num = data[measure_cols].isnull().sum(axis=1)
print("Rows with >= 3 missing measurements:", int((n_missing_num >= 3).sum()))
data = data[n_missing_num < 3].reset_index(drop=True)
print("Shape:", data.shape)
data.isnull().sum()

Rows with >= 3 missing measurements: 2
Shape: (342, 7)


,0
species,0
island,0
bill_length_mm,0
bill_depth_mm,0
flipper_length_mm,0
body_mass_g,0
sex,9


In [12]:
num_cols = data.select_dtypes(include=np.number).columns
num_missing_before = data[num_cols].isnull().sum()

for col in num_cols:
    data[col] = data[col].fillna(data[col].median())

pd.DataFrame({
    "Missing before": num_missing_before,
    "Missing after": data[num_cols].isnull().sum(),
})

,Missing before,Missing after
bill_length_mm,0,0
bill_depth_mm,0,0
flipper_length_mm,0,0
body_mass_g,0,0


In [13]:
cat_cols = data.select_dtypes(include=["object", "category"]).columns
cat_missing_before = data[cat_cols].isnull().sum()

for col in cat_cols:
    if data[col].isnull().any():
        data[col] = data[col].fillna(data[col].mode()[0])

pd.DataFrame({
    "Missing before": cat_missing_before,
    "Missing after": data[cat_cols].isnull().sum(),
})

,Missing before,Missing after
species,0,0
island,0,0
sex,9,0


In [14]:
Q1 = data[num_cols].quantile(0.25)
Q3 = data[num_cols].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_count = {}
for col in num_cols:
    mask = (data[col] < lower_bound[col]) | (data[col] > upper_bound[col])
    outlier_count[col] = int(mask.sum())

pd.DataFrame({
    "Q1": Q1, "Q3": Q3, "IQR": IQR,
    "Lower bound": lower_bound, "Upper bound": upper_bound,
    "Outliers": pd.Series(outlier_count),
})

,Q1,Q3,IQR,Lower bound,Upper bound,Outliers
bill_length_mm,39.225,48.5,9.275,25.3125,62.4125,0
bill_depth_mm,15.600,18.7,3.100,10.9500,23.3500,0
flipper_length_mm,190.000,213.0,23.000,155.5000,247.5000,0
body_mass_g,3550.000,4750.0,1200.000,1750.0000,6550.0000,0


In [15]:
data_capped = data.copy()
for col in num_cols:
    data_capped[col] = data_capped[col].clip(lower=lower_bound[col], upper=upper_bound[col])

outside_after = {
    col: int(((data_capped[col] < lower_bound[col]) | (data_capped[col] > upper_bound[col])).sum())
    for col in num_cols
}
print("Outliers before capping:", outlier_count)
print("Outliers after capping: ", outside_after)

Outliers before capping: {'bill_length_mm': 0, 'bill_depth_mm': 0, 'flipper_length_mm': 0, 'body_mass_g': 0}
Outliers after capping:  {'bill_length_mm': 0, 'bill_depth_mm': 0, 'flipper_length_mm': 0, 'body_mass_g': 0}


In [16]:
standardized_data = data_capped.copy()
standardized_data[num_cols] = StandardScaler().fit_transform(data_capped[num_cols])
standardized_data[num_cols].head()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,-0.884499,0.785449,-1.418347,-0.564142
1,-0.811126,0.126188,-1.062250,-0.501703
2,-0.664380,0.430462,-0.421277,-1.188532
3,-1.324737,1.089724,-0.563715,-0.938776
4,-0.847812,1.748985,-0.777373,-0.689020


In [17]:
final_data = pd.get_dummies(
    standardized_data, columns=["island", "sex"], dtype=int
)
print("Remaining missing values:", int(final_data.isnull().sum().sum()))
print("Shape:", final_data.shape)
final_data.head()

Remaining missing values: 0
Shape: (342, 10)


,species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,island_Biscoe,island_Dream,island_Torgersen,sex_Female,sex_Male
0,Adelie,-0.884499,0.785449,-1.418347,-0.564142,0,0,1,0,1
1,Adelie,-0.811126,0.126188,-1.062250,-0.501703,0,0,1,1,0
2,Adelie,-0.664380,0.430462,-0.421277,-1.188532,0,0,1,1,0
3,Adelie,-1.324737,1.089724,-0.563715,-0.938776,0,0,1,1,0
4,Adelie,-0.847812,1.748985,-0.777373,-0.689020,0,0,1,0,1


In [20]:
from google.colab import files

final_data.to_csv("penguins_clean.csv", index=False, sep=";", decimal=",")
files.download("penguins_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>